In [6]:
import os
from pathlib import Path

import pandas as pd

## The following script aggregates AMI loads for each GISID feeder node.

In [3]:
import os
from pathlib import Path

import pandas as pd

# ==============================================================
# PATHS
# ==============================================================

current_path = Path(os.getcwd())
ami_folder = current_path / ".." / "NREL"

ind_load_csv = ami_folder / "consumption" / "ch1_4301002_individual.csv"
location_ref = ami_folder / "location_reference.csv"
results_folder = current_path / ".." / "results"
results_folder.mkdir(parents=True, exist_ok=True)

print("Using paths:")
print(f"  individual AMI: {ind_load_csv}")
print(f"  location ref  : {location_ref}")
print(f"  results folder: {results_folder}")

# ==============================================================
# READ INPUTS
# ==============================================================

loc = pd.read_csv(location_ref, dtype={"servicepointid": str})
ind = pd.read_csv(ind_load_csv, dtype={"SERVICEPOINTID": str})


# ==============================================================
# FILL MISSING 15-MIN INTERVALS PER SERVICEPOINTID (VERY SIMPLE)
# ==============================================================

# 1. Make ENDTIME_EST a proper datetime
ind["ENDTIME_EST"] = pd.to_datetime(ind["ENDTIME_EST"], errors="coerce")

# 2. Keep only 2024 data
start_date = pd.Timestamp("2024-01-01 00:00:00")
end_date   = pd.Timestamp("2025-01-01 00:00:00")  # end is exclusive


ind = ind[(ind["ENDTIME_EST"] >= start_date) & (ind["ENDTIME_EST"] < end_date)].copy()

# 3. Full 15-min timeline for 2024
full_time = pd.date_range(
    start=start_date,
    end=end_date - pd.Timedelta(minutes=15),
    freq="15min",
)
# REMOVE FEB 29 FROM DATA
full_time = full_time[full_time.date != pd.to_datetime("2024-02-29").date()]

filled_all = []
gap_summary = []

# 4. Loop over each service point ID
for spid, g in ind.groupby("SERVICEPOINTID"):
    # Sort by time
    g = g.sort_values("ENDTIME_EST")

    # --- NEW STEP: remove duplicates at the same timestamp ---
    # If the same SPID has multiple rows at the same ENDTIME_EST,
    # sum the energy and keep the first channel number.
    g = (
        g.groupby("ENDTIME_EST", as_index=False)
         .agg({
             "CHANNELNUMBER": "first",
             "INTERVAL_READ": "sum",
         })
    )

    # Use time as the index
    g = g.set_index("ENDTIME_EST")

    # Reindex to the full 15-min grid
    g2 = g.reindex(full_time)

    # Count how many intervals were missing
    n_missing = g2["INTERVAL_READ"].isna().sum()
    if n_missing > 0:
        gap_summary.append({
            "SERVICEPOINTID": spid,
            "missing_intervals": int(n_missing),
        })

    # Fill in ID and channel
    g2["SERVICEPOINTID"] = spid
    g2["CHANNELNUMBER"] = g2["CHANNELNUMBER"].fillna(1)

    # Fill missing readings with 0
    g2["INTERVAL_READ"] = g2["INTERVAL_READ"].fillna(0)

    # Bring ENDTIME_EST back as a normal column
    g2 = g2.reset_index().rename(columns={"index": "ENDTIME_EST"})

    filled_all.append(g2)

# 5. Put all service points back together
ind = pd.concat(filled_all, ignore_index=True)

# 6. Simple summary of which SPIDs had gaps
if gap_summary:
    gaps_df = pd.DataFrame(gap_summary)
    gaps_path = results_folder / "servicepoints_with_gaps_simple.csv"
    gaps_df.to_csv(gaps_path, index=False)
    print(f"Saved simple gap summary to: {gaps_path}")
else:
    print("No missing intervals found.")

## NOW MAP TO TRANSFORMER IDS
# Clean transformer IDs
loc["xfr_gisid"] = pd.to_numeric(loc["xfr_gisid"], errors="coerce")
loc = loc.dropna(subset=["xfr_gisid"]).copy()
loc["xfr_gisid"] = loc["xfr_gisid"].astype(int)

# ==============================================================
# FIND SERVICEPOINTS IN loc THAT NEVER APPEAR IN ind
# ==============================================================

missing_ids = loc[~loc["servicepointid"].isin(ind["SERVICEPOINTID"])].copy()
missing_ids = missing_ids[["servicepointid", "xfr_gisid", "Residential"]]
missing_ids["Residential"] = missing_ids["Residential"].astype(str).str.strip()

missing_spid_path = results_folder / "missing_service_ids.csv"
missing_ids.to_csv(missing_spid_path, index=False)

print(f"\nSaved missing servicepoints to: {missing_spid_path}")
print(f"Total missing SPIDs: {len(missing_ids)}")
print(missing_ids.head())

if not missing_ids.empty:
    print("\n=== Missing SPIDs by transformer & Residential flag ===")
    print(missing_ids.groupby(["xfr_gisid", "Residential"])["servicepointid"].count())

# ==============================================================
# AGGREGATE LOADS BY TRANSFORMER AND RES / NON-RES
# ==============================================================

all_results = []   # for validation / totals
all_long = []      # for final pivot

for gsid in loc["xfr_gisid"].unique():
    subset = loc[loc["xfr_gisid"] == gsid]

    # Residential / Non-res IDs from location_ref
    _res_mask = subset["Residential"].astype(str).str.strip().str.lower().str.startswith("res")
    res_ids = subset.loc[_res_mask, "servicepointid"].astype(str)
    nonres_ids = subset.loc[~_res_mask, "servicepointid"].astype(str)

    # Only keep IDs that actually appear in AMI data
    res_ids_present = res_ids[res_ids.isin(ind["SERVICEPOINTID"])].unique()
    nonres_ids_present = nonres_ids[nonres_ids.isin(ind["SERVICEPOINTID"])].unique()

    frames = []

    # Aggregate RES if there is at least one SPID with data
    if len(res_ids_present) > 0:
        df_res = ind[ind["SERVICEPOINTID"].isin(res_ids_present)]
        res_agg = df_res.groupby("ENDTIME_EST", as_index=False)["INTERVAL_READ"].sum()
        res_agg.rename(columns={"INTERVAL_READ": "kWh_res"}, inplace=True)
        frames.append(res_agg)

    # Aggregate NON-RES if there is at least one SPID with data
    if len(nonres_ids_present) > 0:
        df_non_res = ind[ind["SERVICEPOINTID"].isin(nonres_ids_present)]
        nonres_agg = df_non_res.groupby("ENDTIME_EST", as_index=False)["INTERVAL_READ"].sum()
        nonres_agg.rename(columns={"INTERVAL_READ": "kWh_nonres"}, inplace=True)
        frames.append(nonres_agg)

    # If neither res nor non-res have any SPID with data, skip this transformer
    if not frames:
        continue

    # Merge the available sector frames on ENDTIME_EST
    df_gsid = frames[0]
    for extra in frames[1:]:
        df_gsid = df_gsid.merge(extra, on="ENDTIME_EST", how="outer")

    df_gsid["xfr_gisid"] = gsid

    # Fill only the columns that actually exist here
    cols_present = [c for c in ["kWh_res", "kWh_nonres"] if c in df_gsid.columns]
    df_gsid[cols_present] = df_gsid[cols_present].fillna(0)

    all_results.append(df_gsid)

    # Build long-format rows for this transformer only for existing sectors
    df_long_gsid = df_gsid.melt(
        id_vars=["ENDTIME_EST", "xfr_gisid"],
        value_vars=cols_present,
        var_name="sector",
        value_name="kWh",
    )

    # Map sector names to "RES" / "NRS"
    sector_map = {"kWh_res": "RES", "kWh_nonres": "NRS"}
    df_long_gsid["sector"] = df_long_gsid["sector"].map(sector_map)

    # Column name: 220603575_Res, 220603575_Non-Res, etc.
    df_long_gsid["col"] = (
        df_long_gsid["xfr_gisid"].astype(int).astype(str) + "_" + df_long_gsid["sector"]
    )

    all_long.append(df_long_gsid)

if not all_results:
    raise RuntimeError("No transformers with load data were found in AMI file.")

df_final = pd.concat(all_results, ignore_index=True)
df_long = pd.concat(all_long, ignore_index=True)

# ==============================================================
# FILTER TO 2024 ONLY (APPLY TO BOTH TABLES)
# ==============================================================

df_final["ENDTIME_EST"] = pd.to_datetime(df_final["ENDTIME_EST"], errors="coerce")
df_long["ENDTIME_EST"] = pd.to_datetime(df_long["ENDTIME_EST"], errors="coerce")

start_date = pd.Timestamp("2024-01-01 00:00:00")
end_date = pd.Timestamp("2025-01-01 00:00:00")

mask_2024 = (df_final["ENDTIME_EST"] >= start_date) & (df_final["ENDTIME_EST"] < end_date)
df_final = df_final[mask_2024].copy()

mask_2024_long = (df_long["ENDTIME_EST"] >= start_date) & (df_long["ENDTIME_EST"] < end_date)
df_long = df_long[mask_2024_long].copy()

# ==============================================================
# LONG → WIDE OUTPUT (OUTPUT FORMAT)
# ==============================================================

# Pivot to wide
df_out = (
    df_long.pivot_table(
        index="ENDTIME_EST", columns="col", values="kWh", aggfunc="sum"
    ).reset_index()
)

df_out.columns.name = None

# Fill any remaining NaNs with 0 in the final output
df_out = df_out.fillna(0)

# Sort non-time columns
other_cols = [c for c in df_out.columns if c != "ENDTIME_EST"]
df_out = df_out[["ENDTIME_EST"] + sorted(other_cols)]

# ==============================================================
# SAVE OUTPUT
# ==============================================================

output_path = results_folder / "aggregated_load_nodes_wide.csv"
df_out.to_csv(output_path, index=False)

# REMOVE FEB 29 FROM DATA
df_out = df_out[df_out["ENDTIME_EST"].dt.date != pd.to_datetime("2024-02-29").date()]

print(f"\nSaved aggregated results (OUTPUT format) to: {output_path}")
print(df_out.head())


# ============================================
# SIMPLE CHECKS FOR TRANSFORMERS IN OUTPUT
# ============================================

# All transformer IDs in location_ref
all_gsid_loc = set(loc["xfr_gisid"].unique())

# Transformer IDs that appear in the output file (from column names)
xfr_in_output = {
    int(c.split("_")[0]) 
    for c in df_out.columns 
    if c != "ENDTIME_EST"
}

# 1. GSIDs that exist in location_ref but NOT in the output file
missing_gsid = sorted(all_gsid_loc - xfr_in_output)

# 2. Total number of GSID_RES / GSID_NRS columns
num_output_cols = len([c for c in df_out.columns if c != "ENDTIME_EST"])

# 3. Total unique GSIDs in the output
num_unique_gsid = len(xfr_in_output)

print("\n=== SIMPLE SUMMARY ===")
print("GSIDs in loc but missing from output:", missing_gsid)
print("Total GSID_RES/NRS columns in output:", num_output_cols)
print("Total unique GSIDs in output:", num_unique_gsid)
print("Total unique GSIDs in loc:", len(all_gsid_loc))

# ==============================================================
# (Optional) consistency check: raw vs aggregated totals for 2024
# ==============================================================

ind["ENDTIME_EST"] = pd.to_datetime(ind["ENDTIME_EST"], errors="coerce")
ind_2024 = ind[
    (ind["ENDTIME_EST"] >= start_date) & (ind["ENDTIME_EST"] < end_date)
]

## CHECKS
# How many unique timestamps do we actually have in 2024?
print("Unique timestamps in 2024:", ind_2024["ENDTIME_EST"].nunique())

# Check the min and max timestamps
print("Min timestamp:", ind_2024["ENDTIME_EST"].min())
print("Max timestamp:", ind_2024["ENDTIME_EST"].max())

# Look at the distribution of time deltas to see if it is always 15 min
ts_sorted = ind_2024["ENDTIME_EST"].sort_values().dropna().unique()
ts_series = pd.Series(ts_sorted)
deltas = ts_series.diff().value_counts()
print("\nTime step distribution:")
print(deltas.head(10))


mapping = loc[["servicepointid", "xfr_gisid", "Residential"]].rename(
    columns={"servicepointid": "SERVICEPOINTID"}
)
joined = ind_2024.merge(mapping, on="SERVICEPOINTID", how="left")

print("\n=== Per Transformer Totals (2024) ===")
for gsid in sorted(df_final["xfr_gisid"].unique()):
    raw_rows = joined[joined["xfr_gisid"] == gsid]
    agg_rows = df_final[df_final["xfr_gisid"] == gsid]

    raw_total = raw_rows["INTERVAL_READ"].sum()
    agg_total = agg_rows.get("kWh_res", 0).sum() + agg_rows.get("kWh_nonres", 0).sum()
    diff = agg_total - raw_total

    print(f"Transformer {gsid}: RAW = {raw_total:.3f}, AGG = {agg_total:.3f}, DIFF = {diff:.3f}")

# For overall totals, only count rows that have a mapped transformer
joined_mapped = joined[~joined["xfr_gisid"].isna()]
raw_total_all = joined_mapped["INTERVAL_READ"].sum()
agg_total_all = df_final.get("kWh_res", 0).sum() + df_final.get("kWh_nonres", 0).sum()

print("\n=== Overall Totals (2024, only mapped SPIDs) ===")
print(f"RAW Total Energy: {raw_total_all:.3f}")
print(f"AGG Total Energy: {agg_total_all:.3f}")

if abs(raw_total_all - agg_total_all) < 1e-3:
    print("Aggregated results match raw totals for 2024.")
else:
    print("Aggregated and raw totals differ — see per-transformer differences above.")


# note
# there are 8 service point IDs missing
# there were no service point IDs missing when all years data was used. When filtered down to 2024, then these IDs were missing


Using paths:
  individual AMI: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../NREL/consumption/ch1_4301002_individual.csv
  location ref  : /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../NREL/location_reference.csv
  results folder: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results
Saved simple gap summary to: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results/servicepoints_with_gaps_simple.csv

Saved missing servicepoints to: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results/missing_service_ids.csv
Total missing SPIDs: 8
    servicepointid  xfr_gisid Residential
216     6000989800  220270646         Res
225     6000991593  220270625         Res
232     6000993734  220268180         Res
398     6000991593  220270627         Res
476     6000999010  220269836         Res

=== Missing SPIDs by transformer & Residential flag ===
xfr_gisid  Residential
220268180  

# Script for aggregating ResStock models for feeder nodes

In [7]:
# read the resstock model to ami match report

# to existing logic, add the resstock model associated with the ami for each feeder node

# Add load profile of the resstock model

# meter id	ResStock_ids
# 6000973858	[13761]
# 6000973865	[139488]

# Read match report
current_path = Path(os.getcwd())
results_folder = current_path / ".." / "results"
match_path = results_folder / "match_report_acheatingwaterheatinginference.csv"

df_match = pd.read_csv(match_path)


ami_folder = current_path / ".." / "NREL"
ind_load_csv = ami_folder / "consumption" / "ch1_4301002_individual.csv"

ind = pd.read_csv(ind_load_csv, dtype={"SERVICEPOINTID": str})

data_dir_resstock = Path('/Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/Files/parquet')

res_ts_path = data_dir_resstock / "resstock_timeseries_all_15.csv"
res_df = pd.read_csv(res_ts_path)


print("df_match:")
print(df_match.head())

print("\nind:")
print(ind.head())

print("\res_df:")
print(res_df.head())

df_match:
     meter id ResStock_ids               cvrmses          cvrmses_month  \
0  6000973858      [13761]  [0.8043237466893888]   [0.2386941743116078]   
1  6000973865     [139488]  [0.8465657569869366]  [0.26783754882306116]   
2  6000973884     [483031]  [1.1766695373203715]  [0.14530217913461768]   
3  6000973888     [182528]  [0.9687557040127385]  [0.11508037731440166]   
4  6000973894      [93383]    [1.44378476859471]  [0.18934979299412055]   

    percent_errs_cooling    percent_errs_heating peak_htg_hr_diffs  \
0  [0.08541846313340556]   [0.05297519987879825]             [2.0]   
1  [0.08826230814844155]   [0.14407488515693614]             [1.0]   
2  [0.04689310362108548]  [0.039254872594854676]             [1.0]   
3  [0.05037544664213248]   [0.05171599117679144]            [11.0]   
4  [0.13419058922109334]   [0.02226686321404066]             [3.0]   

  peak_clg_hr_diffs                    costs  
0             [1.0]  [0.0033453275637905107]  
1             [1.0]   [0

In [4]:
# READ NODE.DSS FILE AND CHECK MISSING SERVICE IDS
#General - BTO Grid Edge\Avangrid - Confidential\Feeder\opendss\Dryden_OpenDSS_Final\test15\load_2.dss

csv_path = Path('/Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/OpenDSS/load_2dss.csv')

columns = ["New", "LoadName", "Bus1", "Phases", "Conn", "kV", "kW", "pf", "Vminpu", "Model"]
df = pd.read_csv(csv_path, header=None, names=columns)

# Clean load names
df["CleanLoadName"] = df["LoadName"].str.replace("Load.", "", regex=False)

# Extract GISID (before underscore)
df["GISID"] = df["CleanLoadName"].str.extract(r"(\d+)").astype(int)


df.head()

AttributeError: Can only use .str accessor with string values!

In [9]:
current_path = Path(os.getcwd())
ami_folder = current_path / ".." / "NREL"

location_ref = ami_folder / "location_reference.csv"

results_folder = current_path / ".." / "results"
results_folder.mkdir(parents=True, exist_ok=True)


loc = pd.read_csv(location_ref, dtype={"servicepointid": str})
xfr_gisid = loc['xfr_gisid']
# THERE IS ONLY ONE NA DROP THAT
# servicepointid	circuit	xfr_gisid	amimeter	Residential	latitude	longitude
# 6003668337	4301002	NULL	Y	Res	42.49059102	-76.28885636

loc = loc.dropna(subset=["xfr_gisid"])
loc["xfr_gisid"] = loc["xfr_gisid"].astype(float).astype(int)

loc.head()

,servicepointid,circuit,xfr_gisid,amimeter,Residential,latitude,longitude
0,6000990168,4301002,220603575,Y,Res,42.491198,-76.299784
1,6000981995,4301002,220270492,Y,Res,42.494522,-76.304065
2,6000998944,4301002,220269834,Y,Res,42.491739,-76.273444
3,6000990452,4301002,220269968,Y,Res,42.497168,-76.279103
4,6000981895,4301002,220270495,Y,Res,42.494890,-76.305322


In [10]:
# Find DSS GISIDs not in loc
missing_gisids = df[~df["GISID"].isin(loc["xfr_gisid"])]["GISID"].unique()

print("GISIDs in DSS but NOT in location_reference:")
print(missing_gisids)

KeyError: 'GISID'

In [ ]:
# MISSING IDS

# total missing IDs that are in load.dss testing but not created
# ['312100149_NRS', '220270217_NRS', '220269836_RES', '312064905_NRS', '220270546_NRS', '220270690_RES', '220270547_NRS', '220270545_NRS', '220270272_RES', '220270701_NRS', '220270688_RES', '220270214_RES', '220270689_RES']

# These dont exist in location ref
# ['312100149_NRS'	 '220270217_NRS'	 '220269836_RES'	 '312064905_NRS'	 '220270546_NRS'	 '220270690_RES'	 '220270547_NRS'	 '220270545_NRS'	 '220270272_RES'	 '220270701_NRS'	 '220270688_RES'	 '220270214_RES'	 '220270689_RES']


# These are in the Location Ref but dont have 2024 AMI data
#ids = ['220269836_RES', '220270701_NRS']

## SCRIPT FOR AGGREGATING RESSTOCK IDS

In [11]:
import os
from pathlib import Path
import pandas as pd
import ast


In [12]:
current_path = Path(os.getcwd())

ami_folder = current_path / ".." / "NREL"
results_folder = current_path / ".." / "results"
results_folder.mkdir(exist_ok=True)

ind_load_csv = ami_folder / "consumption" / "ch1_4301002_individual.csv"
location_ref = ami_folder / "location_reference.csv"

match_path = results_folder / "match_report_acheatingwaterheatinginference.csv"

data_dir_resstock = Path('/Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/Files/parquet')

res_ts_path = data_dir_resstock / "resstock_timeseries_all_15.csv"

print("AMI file:", ind_load_csv)
print("Location file:", location_ref)
print("Match file:", match_path)
print("ResStock TS file:", res_ts_path)
print("Results folder:", results_folder)


AMI file: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../NREL/consumption/ch1_4301002_individual.csv
Location file: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../NREL/location_reference.csv
Match file: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results/match_report_acheatingwaterheatinginference.csv
ResStock TS file: /Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/Files/parquet/resstock_timeseries_all_15.csv
Results folder: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results


In [13]:
ind = pd.read_csv(ind_load_csv, dtype={"SERVICEPOINTID": str})
loc = pd.read_csv(location_ref, dtype={"servicepointid": str})

print("ind rows:", len(ind))
print("loc rows:", len(loc))
print(ind.head())
print(loc.head())


ind rows: 39030035
loc rows: 792
  SERVICEPOINTID  CHANNELNUMBER          ENDTIME_EST  INTERVAL_READ
0     6000973858              1  2023-06-13 14:00:00          0.057
1     6000973858              1  2023-06-13 14:15:00          0.051
2     6000973858              1  2023-06-13 14:30:00          0.013
3     6000973858              1  2023-06-13 14:45:00          0.031
4     6000973858              1  2023-06-13 15:00:00          0.037
  servicepointid  circuit    xfr_gisid amimeter Residential   latitude  \
0     6000990168  4301002  220603575.0        Y         Res  42.491198   
1     6000981995  4301002  220270492.0        Y         Res  42.494522   
2     6000998944  4301002  220269834.0        Y         Res  42.491739   
3     6000990452  4301002  220269968.0        Y         Res  42.497168   
4     6000981895  4301002  220270495.0        Y         Res  42.494890   

   longitude  
0 -76.299784  
1 -76.304065  
2 -76.273444  
3 -76.279103  
4 -76.305322  


In [14]:
ind["ENDTIME_EST"] = pd.to_datetime(ind["ENDTIME_EST"], errors="coerce")

start_2024 = pd.Timestamp("2024-01-01 00:00:00")
end_2025 = pd.Timestamp("2025-01-01 00:00:00")

ind = ind[(ind["ENDTIME_EST"] >= start_2024) & (ind["ENDTIME_EST"] < end_2025)].copy()

full_time = pd.date_range(
    start=start_2024,
    end=end_2025 - pd.Timedelta(minutes=15),
    freq="15min"
)

full_time = full_time[full_time.date != pd.Timestamp("2024-02-29").date()]

print("Filtered ind rows:", len(ind))
print("Full timeline length:", len(full_time))
print("Timeline starts:", full_time.min())
print("Timeline ends:", full_time.max())


Filtered ind rows: 21380228
Full timeline length: 35040
Timeline starts: 2024-01-01 00:00:00
Timeline ends: 2024-12-31 23:45:00


In [ ]:
# CHECK SPIDS THAT ARE MISSING IN MATCH REPORT DATA IN AMI IND

#6000983249	6003654545	6000993773	6003654546	6000989830	6000984738	6000983606	6003654548	6000978854	6000989800	6003654549	6000991593	6000993734	6000978060	6000978885	6000989893	6000973872	6000991593	6000983470	6000978060	6000981021	6000999010	6000993726	6000983426	6000981021	6000978060	6000978862	6001000956	6000991593	6000981021	6000978869	6000978846	6000978876

spid = [6000983249,	6003654545,	6000993773	6003654546	6000989830	6000984738]


In [15]:
filled_rows = []

for gsid in loc["xfr_gisid"].unique():

    meters = loc.loc[loc["xfr_gisid"] == gsid, "servicepointid"].astype(str).unique()
    # COPY SERVICE POINT ID FROM IN THAT ARE IN LOCATION REF (UNIQUE)
    chunk = ind[ind["SERVICEPOINTID"].isin(meters)].copy()

    if chunk.empty:
        continue
    ## If there are duplicates per (time, meter), sum them
    chunk = (
        chunk.groupby(["SERVICEPOINTID", "ENDTIME_EST"], as_index=False)["INTERVAL_READ"]
             .sum()
    )
    # FIND UNIQUE SERVICE POINT ID
    all_spids = chunk["SERVICEPOINTID"].unique()
    full_index = pd.MultiIndex.from_product([all_spids, full_time],
                                            names=["SERVICEPOINTID", "ENDTIME_EST"])
    # FILL THE NA WITH 0
    chunk_filled = (
        chunk.set_index(["SERVICEPOINTID", "ENDTIME_EST"])
             .reindex(full_index)
             .fillna(0)
             .reset_index()
    )

    filled_rows.append(chunk_filled)

ind_filled = pd.concat(filled_rows, ignore_index=True)


In [16]:
loc["xfr_gisid"] = pd.to_numeric(loc["xfr_gisid"], errors="coerce")
loc = loc.dropna(subset=["xfr_gisid"]).copy()
loc["xfr_gisid"] = loc["xfr_gisid"].astype(int)

print("Unique transformers:", loc["xfr_gisid"].nunique())


Unique transformers: 177


# THIS IS A FASTER VERSION OF THE EARLIER AMI LOAD AGGREGATION SCRIPT AT THE TOP

In [17]:
ami_long_rows = []

for transformer in loc["xfr_gisid"].unique():

    meters = loc[loc["xfr_gisid"] == transformer].copy()

    is_res = meters["Residential"].astype(str).str.lower().str.startswith("res")

    res_meters = meters.loc[is_res, "servicepointid"].astype(str)
    nrs_meters = meters.loc[~is_res, "servicepointid"].astype(str)

    # Only keep meters that exist in the filled AMI data
    res_meters = res_meters[res_meters.isin(ind_filled["SERVICEPOINTID"])].unique()
    nrs_meters = nrs_meters[nrs_meters.isin(ind_filled["SERVICEPOINTID"])].unique()

    # RES
    if len(res_meters) > 0:
        res_data = ind_filled[ind_filled["SERVICEPOINTID"].isin(res_meters)]
        res_sum = res_data.groupby("ENDTIME_EST")["INTERVAL_READ"].sum()

        for t, val in res_sum.items():
            ami_long_rows.append({"ENDTIME_EST": t, "col": f"{transformer}_RES", "kWh": val})

    # NRS
    if len(nrs_meters) > 0:
        nrs_data = ind_filled[ind_filled["SERVICEPOINTID"].isin(nrs_meters)]
        nrs_sum = nrs_data.groupby("ENDTIME_EST")["INTERVAL_READ"].sum()

        for t, val in nrs_sum.items():
            ami_long_rows.append({"ENDTIME_EST": t, "col": f"{transformer}_NRS", "kWh": val})

ami_long = pd.DataFrame(ami_long_rows)

ami_wide = (
    ami_long.pivot(index="ENDTIME_EST", columns="col", values="kWh")
    .fillna(0)
    .reset_index()
)

ami_out_path = results_folder / "aggregated_load_nodes_wide_ami.csv"
ami_wide.to_csv(ami_out_path, index=False)

print("Saved AMI output:", ami_out_path)
print(ami_wide.head())


Saved AMI output: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results/aggregated_load_nodes_wide_ami.csv
col         ENDTIME_EST  220268180_RES  220268181_RES  220268182_RES  \
0   2024-01-01 00:00:00          0.225          0.187            0.0   
1   2024-01-01 00:15:00          0.217          0.123            0.0   
2   2024-01-01 00:30:00          0.246          0.145            0.0   
3   2024-01-01 00:45:00          0.196          0.164            0.0   
4   2024-01-01 01:00:00          0.234          0.127            0.0   

col  220268183_NRS  220268183_RES  220268184_RES  220268252_RES  \
0              0.0          0.356            0.0          0.140   
1              0.0          0.446            0.0          0.198   
2              0.0          0.472            0.0          0.186   
3              0.0          0.778            0.0          0.118   
4              0.0          0.218            0.0          0.109   

col  220268260_NRS  220268260_RES

In [18]:
# READ MATCH REPORT
df_match = pd.read_csv(match_path)

df_match["meter id"] = df_match["meter id"].astype(str)

# Convert "[13761]" to [13761]
df_match["ResStock_ids"] = df_match["ResStock_ids"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)
# MATCH METER TO RESSTOCK
meter_to_resstock = dict(zip(df_match["meter id"], df_match["ResStock_ids"]))

print("Match rows:", len(df_match))
print("Example mapping:", list(meter_to_resstock.items())[:3])


Match rows: 538
Example mapping: [('6000973858', [13761]), ('6000973865', [139488]), ('6000973884', [483031])]


In [19]:
# READ RESSTOCK TIMESERIES
res_df = pd.read_csv(res_ts_path)

res_df["time"] = pd.to_datetime(res_df["time"], errors="coerce")
# STORE 2024
res_df = res_df[(res_df["time"] >= start_2024) & (res_df["time"] < end_2025)].copy()
res_df = res_df[res_df["time"].dt.date != pd.Timestamp("2024-02-29").date()].copy()

print("ResStock rows:", len(res_df))
print(res_df.head())


ResStock rows: 21269280
   bldg_id                time   ts_elec
0     6975 2024-01-01 00:15:00  0.000000
1     6975 2024-01-01 00:30:00  0.045916
2     6975 2024-01-01 00:45:00  0.045916
3     6975 2024-01-01 01:00:00  0.045916
4     6975 2024-01-01 01:15:00  0.045916


In [ ]:
# resstock_long_rows = []

# for transformer in loc["xfr_gisid"].unique():

#     meters = loc[loc["xfr_gisid"] == transformer].copy()

#     is_res = meters["Residential"].astype(str).str.lower().str.startswith("res")

#     res_meters = meters.loc[is_res, "servicepointid"].astype(str).unique()
#     nrs_meters = meters.loc[~is_res, "servicepointid"].astype(str).unique()

#     # Keep only meters that appear in match file
#     res_meters = [m for m in res_meters if m in meter_to_resstock]
#     nrs_meters = [m for m in nrs_meters if m in meter_to_resstock]

#     # Get RES ResStock IDs
#     resstock_ids_res = []
#     for m in res_meters:
#         resstock_ids_res.extend(meter_to_resstock[m])
#     resstock_ids_res = sorted(list(set(resstock_ids_res)))

#     # Get NRS ResStock IDs
#     resstock_ids_nrs = []
#     for m in nrs_meters:
#         resstock_ids_nrs.extend(meter_to_resstock[m])
#     resstock_ids_nrs = sorted(list(set(resstock_ids_nrs)))

#     # RES ResStock aggregation
#     if len(resstock_ids_res) > 0:
#         tmp = res_df[res_df["bldg_id"].isin(resstock_ids_res)]
#         ts_sum = tmp.groupby("time")["ts_elec"].sum()

#         for t, val in ts_sum.items():
#             resstock_long_rows.append({"ENDTIME_EST": t, "col": f"{transformer}_RES", "kWh": val})

#     # NRS ResStock aggregation
#     if len(resstock_ids_nrs) > 0:
#         tmp = res_df[res_df["bldg_id"].isin(resstock_ids_nrs)]
#         ts_sum = tmp.groupby("time")["ts_elec"].sum()

#         for t, val in ts_sum.items():
#             resstock_long_rows.append({"ENDTIME_EST": t, "col": f"{transformer}_NRS", "kWh": val})

# resstock_long = pd.DataFrame(resstock_long_rows)

# resstock_wide = (
#     resstock_long.pivot(index="ENDTIME_EST", columns="col", values="kWh")
#     .fillna(0)
#     .reset_index()
# )

# resstock_out_path = results_folder / "aggregated_load_nodes_wide_resstock.csv"
# resstock_wide.to_csv(resstock_out_path, index=False)

# print("Saved ResStock output:", resstock_out_path)
# print(resstock_wide.head())


# THIS SCRIPTS AGGREGATES RESSTOCK TO NODE

In [25]:
import pandas as pd

start_date = pd.Timestamp("2024-01-01 00:00:00")
end_date   = pd.Timestamp("2025-01-01 00:00:00")

# ==============================================================
# RESSTOCK (RES ONLY): aggregate by transformer + simple checks
# ==============================================================

# ---------- 1) Make sure the required data exists ----------
needed = ["loc", "meter_to_resstock", "res_df", "results_folder", "start_date", "end_date"]
for name in needed:
    if name not in globals():
        raise NameError(f"You must create `{name}` before running this cell.")

# ---------- 2) Clean types so joins actually work ----------
loc = loc.copy()
loc["servicepointid"] = loc["servicepointid"].astype(str)

loc["xfr_gisid"] = pd.to_numeric(loc["xfr_gisid"], errors="coerce")
loc = loc.dropna(subset=["xfr_gisid"]).copy()
loc["xfr_gisid"] = loc["xfr_gisid"].astype(int)

res_df = res_df.copy()
res_df["time"] = pd.to_datetime(res_df["time"], errors="coerce")
res_df["bldg_id"] = pd.to_numeric(res_df["bldg_id"], errors="coerce")
res_df = res_df.dropna(subset=["time", "bldg_id"]).copy()
res_df["bldg_id"] = res_df["bldg_id"].astype(int)

# ---------- 3) Keep only 2024 and remove Feb 29 ----------
res_2024 = res_df[(res_df["time"] >= start_date) & (res_df["time"] < end_date)].copy()
res_2024 = res_2024[res_2024["time"].dt.date != pd.Timestamp("2024-02-29").date()].copy()

# ---------- 4) Keep only RES meters from loc ----------
is_res_meter = loc["Residential"].astype(str).str.strip().str.lower().str.startswith("res")
loc_res = loc[is_res_meter].copy()

# ==============================================================
# CHECK 1: Which RES meters do NOT have a ResStock match?
# ==============================================================

res_meters_all = loc_res["servicepointid"].unique()

res_meters_missing_match = []
for spid in res_meters_all:
    if spid not in meter_to_resstock:
        res_meters_missing_match.append(spid)

missing_match_df = loc_res[loc_res["servicepointid"].isin(res_meters_missing_match)][
    ["servicepointid", "xfr_gisid", "Residential"]
].copy()

missing_match_path = results_folder / "missing_res_service_ids_in_match_report.csv"
missing_match_df.to_csv(missing_match_path, index=False)

print("\nRES meters missing from match_report:", len(missing_match_df))
print("Saved to:", missing_match_path)
print(missing_match_df.head())

# ==============================================================
# BUILD: ResStock RES load per transformer
# ==============================================================

resstock_rows = []   # will become a long table: time, transformer_RES, kWh

for transformer_id in loc["xfr_gisid"].unique():

    # (a) all RES meters on this transformer
    meters_on_this_xfr = loc_res[loc_res["xfr_gisid"] == transformer_id]["servicepointid"].unique()

    # (b) keep only meters that exist in meter_to_resstock
    meters_with_match = []
    for m in meters_on_this_xfr:
        if m in meter_to_resstock:
            meters_with_match.append(m)

    # (c) collect ALL resstock building IDs for those meters
    resstock_ids = []
    for m in meters_with_match:
        resstock_ids.extend(meter_to_resstock[m])

    # remove duplicates
    resstock_ids = sorted(set(resstock_ids))

    # if nothing matched, skip this transformer
    if len(resstock_ids) == 0:
        continue

    # (d) pull those buildings from ResStock and sum per time
    these_buildings = res_2024[res_2024["bldg_id"].isin(resstock_ids)]
    summed = these_buildings.groupby("time")["ts_elec"].sum()

    # (e) store results as long rows
    for t, kwh in summed.items():
        resstock_rows.append({
            "ENDTIME_EST": t,
            "col": f"{transformer_id}_RES",
            "kWh": kwh
        })

# turn list of rows into a dataframe
resstock_long = pd.DataFrame(resstock_rows)

if resstock_long.empty:
    raise RuntimeError("No RES ResStock data was produced. Check match_report and res_df.")

# ==============================================================
# Convert long -> wide (same format as AMI output)
# ==============================================================

resstock_wide = (
    resstock_long.pivot(index="ENDTIME_EST", columns="col", values="kWh")
    .fillna(0)
    .reset_index()
)

# Save RES-only output
resstock_out_path = results_folder / "aggregated_load_nodes_wide_resstock_RES_ONLY.csv"
resstock_wide.to_csv(resstock_out_path, index=False)

print("\nSaved RES-only ResStock output to:", resstock_out_path)
print(resstock_wide.head())

# ==============================================================
# CHECK 2: Which transformers are missing from the output?
# ==============================================================

all_transformers_in_loc = set(loc["xfr_gisid"].unique())

transformers_in_output = set()
for c in resstock_wide.columns:
    if c == "ENDTIME_EST":
        continue
    transformers_in_output.add(int(c.split("_")[0]))

missing_transformers = sorted(all_transformers_in_loc - transformers_in_output)

print("\nTransformers in loc:", len(all_transformers_in_loc))
print("Transformers in RES ResStock output:", len(transformers_in_output))
print("Transformers missing from RES ResStock output:", missing_transformers)

# ==============================================================
# CHECK 3: ResStock timestamps look like 15-minute data
# ==============================================================

print("\nResStock time checks (2024, Feb 29 removed):")
print("Unique timestamps:", res_2024["time"].nunique())
print("Min time:", res_2024["time"].min())
print("Max time:", res_2024["time"].max())

sorted_times = res_2024["time"].sort_values().dropna().unique()
time_deltas = pd.Series(sorted_times).diff().value_counts()
print("\nTime step distribution (top 10):")
print(time_deltas.head(10))

# ==============================================================
# CHECK 4: Per-transformer totals (RAW vs AGG) should match
# ==============================================================

print("\nPer-transformer totals check (RES only):")

# Precompute transformer -> resstock IDs (RES only)
xfr_to_resstock_ids = {}

for transformer_id in loc["xfr_gisid"].unique():

    meters_on_this_xfr = loc_res[loc_res["xfr_gisid"] == transformer_id]["servicepointid"].unique()

    ids = []
    for m in meters_on_this_xfr:
        if m in meter_to_resstock:
            ids.extend(meter_to_resstock[m])

    xfr_to_resstock_ids[transformer_id] = sorted(set(ids))

# Compare RAW vs AGG for each transformer in output
for transformer_id in sorted(transformers_in_output):

    ids = xfr_to_resstock_ids.get(transformer_id, [])

    raw_total = res_2024[res_2024["bldg_id"].isin(ids)]["ts_elec"].sum()

    col_name = f"{transformer_id}_RES"
    agg_total = resstock_wide[col_name].sum() if col_name in resstock_wide.columns else 0

    print(f"Transformer {transformer_id}: RAW={raw_total:.3f}  AGG={agg_total:.3f}  DIFF={(agg_total-raw_total):.3f}")

# Overall totals across output transformers
raw_total_all = 0
for transformer_id in transformers_in_output:
    ids = xfr_to_resstock_ids.get(transformer_id, [])
    raw_total_all += res_2024[res_2024["bldg_id"].isin(ids)]["ts_elec"].sum()

agg_total_all = 0
for c in resstock_wide.columns:
    if c != "ENDTIME_EST":
        agg_total_all += resstock_wide[c].sum()

print("\nOverall totals check (RES only):")
print(f"RAW total (mapped transformers): {raw_total_all:.3f}")
print(f"AGG total: {agg_total_all:.3f}")
print(f"DIFF: {(agg_total_all-raw_total_all):.3f}")



RES meters missing from match_report: 33
Saved to: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results/missing_res_service_ids_in_match_report.csv
   servicepointid  xfr_gisid Residential
11     6000983249  310007575         Res
20     6003654545  312292599         Res
43     6000993773  220268182         Res
55     6003654546  312292599         Res
94     6000989830  220603575         Res

Saved RES-only ResStock output to: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results/aggregated_load_nodes_wide_resstock_RES_ONLY.csv
col         ENDTIME_EST  220268180_RES  220268181_RES  220268183_RES  \
0   2024-01-01 00:00:00       0.281185       0.147639       1.171721   
1   2024-01-01 00:15:00       0.000000       0.000000       0.000000   
2   2024-01-01 00:30:00       0.150365       0.146244       0.881496   
3   2024-01-01 00:45:00       0.150607       0.145657       0.866862   
4   2024-01-01 01:00:00       0.150454       0.144834    

# THIS SCRIPT ADDS AMI IF RESSTOCK IS MISSING

In [20]:
import pandas as pd

start_date = pd.Timestamp("2024-01-01 00:00:00")
end_date   = pd.Timestamp("2025-01-01 00:00:00")

# ==============================================================
# RESSTOCK (RES ONLY): aggregate by transformer + simple checks
# ==============================================================

# ---------- 1) Make sure the required data exists ----------
needed = ["loc", "meter_to_resstock", "res_df", "results_folder", "start_date", "end_date"]
for name in needed:
    if name not in globals():
        raise NameError(f"You must create `{name}` before running this cell.")

# ---------- 2) Clean types so joins actually work ----------
loc = loc.copy()
loc["servicepointid"] = loc["servicepointid"].astype(str)

loc["xfr_gisid"] = pd.to_numeric(loc["xfr_gisid"], errors="coerce")
loc = loc.dropna(subset=["xfr_gisid"]).copy()
loc["xfr_gisid"] = loc["xfr_gisid"].astype(int)

res_df = res_df.copy()
res_df["time"] = pd.to_datetime(res_df["time"], errors="coerce")
res_df["bldg_id"] = pd.to_numeric(res_df["bldg_id"], errors="coerce")
res_df = res_df.dropna(subset=["time", "bldg_id"]).copy()
res_df["bldg_id"] = res_df["bldg_id"].astype(int)

# ---------- 3) Keep only 2024 and remove Feb 29 ----------
res_2024 = res_df[(res_df["time"] >= start_date) & (res_df["time"] < end_date)].copy()
res_2024 = res_2024[res_2024["time"].dt.date != pd.Timestamp("2024-02-29").date()].copy()

# ---------- 4) Keep only RES meters from loc ----------
is_res_meter = loc["Residential"].astype(str).str.strip().str.lower().str.startswith("res")
loc_res = loc[is_res_meter].copy()

# ==============================================================
# CHECK 1: Which RES meters do NOT have a ResStock match?
# ==============================================================

res_meters_all = loc_res["servicepointid"].unique()

res_meters_missing_match = []
for spid in res_meters_all:
    if spid not in meter_to_resstock:
        res_meters_missing_match.append(spid)

missing_match_df = loc_res[loc_res["servicepointid"].isin(res_meters_missing_match)][
    ["servicepointid", "xfr_gisid", "Residential"]
].copy()

missing_match_path = results_folder / "missing_res_service_ids_in_match_report.csv"
missing_match_df.to_csv(missing_match_path, index=False)

print("\nRES meters missing from match_report:", len(missing_match_df))
print("Saved to:", missing_match_path)
print(missing_match_df.head())


# ==============================================================
# BUILD AMI WIDE (for filling missing ResStock meters)
# ==============================================================
# Clean AMI
ind = ind.copy()
ind["SERVICEPOINTID"] = ind["SERVICEPOINTID"].astype(str)
ind["INTERVAL_READ"] = pd.to_numeric(ind["INTERVAL_READ"], errors="coerce")
ind = ind.dropna(subset=["SERVICEPOINTID", "ENDTIME_EST", "INTERVAL_READ"])

# Keep 2024 only, remove Feb 29
ind = ind[(ind["ENDTIME_EST"] >= start_date) & (ind["ENDTIME_EST"] < end_date)]
ind = ind[ind["ENDTIME_EST"].dt.date != pd.Timestamp("2024-02-29").date()]

# Sum duplicates
ind = ind.groupby(
    ["ENDTIME_EST", "SERVICEPOINTID"], as_index=False
)["INTERVAL_READ"].sum()

# Pivot to wide
ami_wide = (
    ind.pivot(index="ENDTIME_EST", columns="SERVICEPOINTID", values="INTERVAL_READ")
    .fillna(0)
    .reset_index()
)

print("AMI wide shape:", ami_wide.shape)


# ==============================================================
# BUILD: RES per transformer (ResStock first, AMI if missing)
# ==============================================================
rows = []

# identify RES meters
loc["Residential"] = loc["Residential"].astype(str).str.lower()
loc_res = loc[loc["Residential"].str.startswith("res")].copy()
loc_res["xfr_gisid"] = pd.to_numeric(loc_res["xfr_gisid"], errors="coerce")
loc_res = loc_res.dropna(subset=["xfr_gisid"])
loc_res["xfr_gisid"] = loc_res["xfr_gisid"].astype(int)

for xfr in loc_res["xfr_gisid"].unique():

    meters = loc_res[loc_res["xfr_gisid"] == xfr]["servicepointid"].unique()

    meters_with_rs = [m for m in meters if m in meter_to_resstock]
    meters_missing = [m for m in meters if m not in meter_to_resstock]

    # --- ResStock ---
    rs_ids = []
    for m in meters_with_rs:
        rs_ids.extend(meter_to_resstock[m])
    rs_ids = list(set(rs_ids))

    if rs_ids:
        rs = (
            res_2024[res_2024["bldg_id"].isin(rs_ids)]
            .groupby("time")["ts_elec"]
            .sum()
        )
    else:
        rs = pd.Series(dtype=float)

    # --- AMI (ONLY missing meters) ---
    ami_cols = [m for m in meters_missing if m in ami_wide.columns]
    if ami_cols:
        ami = ami_wide.set_index("ENDTIME_EST")[ami_cols].sum(axis=1)
    else:
        ami = pd.Series(dtype=float)

    total = rs.add(ami, fill_value=0)

    if total.empty:
        continue

    for t, v in total.items():
        rows.append({
            "ENDTIME_EST": t,
            "col": f"{xfr}_RES",
            "kWh": v
        })


# ==============================================================
# SAVE filled RES output
# ==============================================================
res_filled = (
    pd.DataFrame(rows)
    .pivot(index="ENDTIME_EST", columns="col", values="kWh")
    .fillna(0)
    .reset_index()
)

out_path = Path(
    "/Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/results/"
    "aggregated_load_nodes_wide_resstock_RES_ONLY_resstock_filled_missingami.csv"
)

res_filled.to_csv(out_path, index=False)

print("Saved:", out_path)
print(res_filled.head())




# # ==============================================================
# # Convert long -> wide (same format as AMI output)
# # ==============================================================

# resstock_wide = (
#     resstock_long.pivot(index="ENDTIME_EST", columns="col", values="kWh")
#     .fillna(0)
#     .reset_index()
# )

# # Save RES-only output
# resstock_out_path = results_folder / "aggregated_load_nodes_wide_resstock_RES_ONLY.csv"
# resstock_wide.to_csv(resstock_out_path, index=False)

# print("\nSaved RES-only ResStock output to:", resstock_out_path)
# print(resstock_wide.head())

# ==============================================================
# CHECK 2: Which transformers are missing from the output?
# ==============================================================

all_transformers_in_loc = set(loc["xfr_gisid"].unique())

transformers_in_output = set()
for c in resstock_wide.columns:
    if c == "ENDTIME_EST":
        continue
    transformers_in_output.add(int(c.split("_")[0]))

missing_transformers = sorted(all_transformers_in_loc - transformers_in_output)

print("\nTransformers in loc:", len(all_transformers_in_loc))
print("Transformers in RES ResStock output:", len(transformers_in_output))
print("Transformers missing from RES ResStock output:", missing_transformers)

# ==============================================================
# CHECK 3: ResStock timestamps look like 15-minute data
# ==============================================================

print("\nResStock time checks (2024, Feb 29 removed):")
print("Unique timestamps:", res_2024["time"].nunique())
print("Min time:", res_2024["time"].min())
print("Max time:", res_2024["time"].max())

sorted_times = res_2024["time"].sort_values().dropna().unique()
time_deltas = pd.Series(sorted_times).diff().value_counts()
print("\nTime step distribution (top 10):")
print(time_deltas.head(10))

# ==============================================================
# CHECK 4: Per-transformer totals (RAW vs AGG) should match
# ==============================================================

print("\nPer-transformer totals check (RES only):")

# Precompute transformer -> resstock IDs (RES only)
xfr_to_resstock_ids = {}

for transformer_id in loc["xfr_gisid"].unique():

    meters_on_this_xfr = loc_res[loc_res["xfr_gisid"] == transformer_id]["servicepointid"].unique()

    ids = []
    for m in meters_on_this_xfr:
        if m in meter_to_resstock:
            ids.extend(meter_to_resstock[m])

    xfr_to_resstock_ids[transformer_id] = sorted(set(ids))

# Compare RAW vs AGG for each transformer in output
for transformer_id in sorted(transformers_in_output):

    ids = xfr_to_resstock_ids.get(transformer_id, [])

    raw_total = res_2024[res_2024["bldg_id"].isin(ids)]["ts_elec"].sum()

    col_name = f"{transformer_id}_RES"
    agg_total = resstock_wide[col_name].sum() if col_name in resstock_wide.columns else 0

    print(f"Transformer {transformer_id}: RAW={raw_total:.3f}  AGG={agg_total:.3f}  DIFF={(agg_total-raw_total):.3f}")

# Overall totals across output transformers
raw_total_all = 0
for transformer_id in transformers_in_output:
    ids = xfr_to_resstock_ids.get(transformer_id, [])
    raw_total_all += res_2024[res_2024["bldg_id"].isin(ids)]["ts_elec"].sum()

agg_total_all = 0
for c in resstock_wide.columns:
    if c != "ENDTIME_EST":
        agg_total_all += resstock_wide[c].sum()

print("\nOverall totals check (RES only):")
print(f"RAW total (mapped transformers): {raw_total_all:.3f}")
print(f"AGG total: {agg_total_all:.3f}")
print(f"DIFF: {(agg_total_all-raw_total_all):.3f}")



RES meters missing from match_report: 33
Saved to: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results/missing_res_service_ids_in_match_report.csv
   servicepointid  xfr_gisid Residential
11     6000983249  310007575         Res
20     6003654545  312292599         Res
43     6000993773  220268182         Res
55     6003654546  312292599         Res
94     6000989830  220603575         Res
AMI wide shape: (34525, 671)
Saved: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/results/aggregated_load_nodes_wide_resstock_RES_ONLY_resstock_filled_missingami.csv
col         ENDTIME_EST  220268180_RES  220268181_RES  220268182_RES  \
0   2024-01-01 00:00:00       0.281185       0.147639            0.0   
1   2024-01-01 00:15:00       0.000000       0.000000            0.0   
2   2024-01-01 00:30:00       0.150365       0.146244            0.0   
3   2024-01-01 00:45:00       0.150607       0.145657            0.0   
4   2024-01-01 01:00:00       0.150454     

NameError: name 'resstock_wide' is not defined

In [26]:
import pandas as pd
import ast

# ---- paste your missing GSIDs here ----
missing_gsids = [
    220268182, 220268184, 220269720, 220269721, 220269722, 220269723, 220269817,
    220269822, 220269836, 220269837, 220269962, 220270381, 220270554, 220270556,
    220270557, 220270629, 220270630, 220270631, 220270642, 220270657, 220270658,
    220270688, 220529264, 220569347, 220609086, 220609092, 310038852, 312292599
]

# Required existing objects:
# loc: columns include ["servicepointid","xfr_gisid","Residential"]
# df_match: columns include ["meter id","ResStock_ids"] (ResStock_ids looks like "[12345]")
# res_df or res_2024: columns include ["bldg_id","time","ts_elec"]
# results_folder: Path

print("loc rows:", len(loc))
print("df_match rows:", len(df_match))


loc rows: 791
df_match rows: 538


In [27]:
# ---------- Clean loc ----------
loc2 = loc.copy()
loc2["servicepointid"] = loc2["servicepointid"].astype(str)

loc2["xfr_gisid"] = pd.to_numeric(loc2["xfr_gisid"], errors="coerce")
loc2 = loc2.dropna(subset=["xfr_gisid"]).copy()
loc2["xfr_gisid"] = loc2["xfr_gisid"].astype(int)

# Keep RES only (because you said ResStock only has RES)
is_res = loc2["Residential"].astype(str).str.strip().str.lower().str.startswith("res")
loc_res = loc2[is_res].copy()

# ---------- Clean df_match ----------
m = df_match.copy()
m["meter id"] = m["meter id"].astype(str)

# Convert string like "[13761]" -> python list [13761]
m["ResStock_ids"] = m["ResStock_ids"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("[") else []
)

# SPID -> list of ResStock IDs
spid_to_resstock = dict(zip(m["meter id"], m["ResStock_ids"]))

print("Unique RES meters in loc:", loc_res["servicepointid"].nunique())
print("Unique meters in match_report:", m["meter id"].nunique())


Unique RES meters in loc: 565
Unique meters in match_report: 538


In [29]:
rows = []

for _, r in loc.iterrows():
    spid = str(r["servicepointid"])
    gsid = r["xfr_gisid"]
    is_res = str(r["Residential"]).lower().startswith("res")

    # Only RES meters matter for ResStock
    if not is_res:
        continue

    # Case 1: SPID exists in match report
    if spid in meter_to_resstock:
        resstock_ids = meter_to_resstock[spid]

        # If mapped but empty list (edge case)
        if len(resstock_ids) == 0:
            rows.append({
                "GSID": gsid,
                "SPID": spid,
                "ResStock_id": None,
                "reason": "SPID in match_report but no ResStock IDs"
            })
        else:
            for rid in resstock_ids:
                rows.append({
                    "GSID": gsid,
                    "SPID": spid,
                    "ResStock_id": rid,
                    "reason": "matched"
                })

    # Case 2: SPID never matched at all
    else:
        rows.append({
            "GSID": gsid,
            "SPID": spid,
            "ResStock_id": None,
            "reason": "SPID missing from match_report"
        })

resstock_mapping_df = pd.DataFrame(rows)

out_path = results_folder / "resstock_spid_gsid_mapping_RES_ONLY.csv"
resstock_mapping_df.to_csv(out_path, index=False)

print("Saved:", out_path)
print(resstock_mapping_df.head(10))


Saved: /Users/tcharan/Documents/grid_edge_ithaca/grid_edge_fy25/notebooks/../results/resstock_spid_gsid_mapping_RES_ONLY.csv
        GSID        SPID  ResStock_id                          reason
0  220603575  6000990168     253262.0                         matched
1  220270492  6000981995     397621.0                         matched
2  220269834  6000998944     443788.0                         matched
3  220269968  6000990452      59156.0                         matched
4  220270495  6000981895     281716.0                         matched
5  220270522  6000999930     139488.0                         matched
6  220270541  6000983779     350191.0                         matched
7  220270565  6000999681     408994.0                         matched
8  220270617  6000984893     361850.0                         matched
9  310007575  6000983249          NaN  SPID missing from match_report
